# Laboratorio 02 · Dai file di dati ai modelli Gurobi

**Ottimizzazione: Modelli e Metodi** · Laurea triennale in Statistica · Università degli Studi di Padova  
**Anthony Palmieri** · Didattica integrativa

In questa lezione formuliamo e risolviamo quattro problemi: **dieta, ripartizione di risorse,
trasporti e portfolio optimization**. Per ciascuno seguiremo lo stesso percorso:
**consegna → tentativo matematico → correzione → tentativo Python → soluzione → interpretazione**.

I dati numerici dei quattro problemi sono letti da veri file CSV con pandas;
anche i risultati vengono salvati su file.

Installiamo solo `gurobipy`.
I modelli hanno al massimo 12 variabili: sono compatibili con la licenza limitata
inclusa nell’installazione pip, per ricerca, sviluppo e valutazione.

In [1]:
%pip -q install gurobipy

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importiamo le librerie principali

from pathlib import Path
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from IPython.display import display

pd.set_option("display.max_columns", 12)
pd.set_option("display.precision", 4)  # cambia soltanto la visualizzazione
TOL = 1e-6
print("pandas:", pd.__version__, "NumPy:", np.__version__, "Gurobi:", gp.gurobi.version())

pandas: 2.1.3 NumPy: 1.26.4 Gurobi: (13, 0, 3)


<a id="ripasso"></a>
## 0. Ripasso operativo di gurobipy

**Comandi principali:**

| Istruzione | Ruolo |
|---|---|
| `gp.Model("nome")` | Crea un modello |
| `m.addVars(indici, lb=0, vtype=...)` | Crea variabili indicizzate |
| `GRB.CONTINUOUS`, `GRB.INTEGER`, `GRB.BINARY` | Reali, intere, binarie |
| `gp.quicksum(...)` | Costruisce una somma di espressioni |
| `m.setObjective(espressione, GRB.MAXIMIZE)` | Definisce l’obiettivo |
| `m.addConstr(espressione <= limite)` | Aggiunge un vincolo |
| `m.optimize()` | Avvia la risoluzione |
| `m.Status`, `m.SolCount` | Stato e numero di soluzioni disponibili |
| `x[j].X`, `m.ObjVal` | Decisioni e obiettivo, **solo se `SolCount > 0`** |

Un generatore come `gp.quicksum(p[j] * x[j] for j in J)` traduce una sommatoria; `==` costruisce un vincolo di uguaglianza; `=` assegna un valore Python.

### Un modello giocattolo

Scriviamo in Gurobi il seguente modello:

$$
\max\;3x_1+5x_2\qquad
\text{s.t. }2x_1+x_2\le8,\quad x_1+2x_2\le8,\quad x_1,x_2\in\mathbb Z_{\ge0}.
$$

In [3]:
# Tocca a voi: create il modello di cui sopra.
# Dichiarate variabili intere non negative, obiettivo e due vincoli.
# Risolvete e controllate SolCount prima di leggere X e ObjVal.

### Correzione del ripasso

`addVars([1, 2], ...)` crea le chiavi 1 e 2.\
`addVars(2, ...)` crea le chiavi `range(2)`\
`OutputFlag = 0` sopprime l’output.\
Una soluzione disponibile non è necessariamente ottima: lo stato `GRB.OPTIMAL` certifica l’ottimalità entro le tolleranze del solver.\
 `assert` segnala una verifica fallita.

In [4]:
m_ripasso = gp.Model("ripasso")
m_ripasso.Params.OutputFlag = 0
x_ripasso = m_ripasso.addVars([1, 2], lb=0, vtype=GRB.INTEGER, name="x")
m_ripasso.setObjective(3*x_ripasso[1] + 5*x_ripasso[2], GRB.MAXIMIZE)
m_ripasso.addConstr(2*x_ripasso[1] + x_ripasso[2] <= 8)
m_ripasso.addConstr(x_ripasso[1] + 2*x_ripasso[2] <= 8)
m_ripasso.optimize()

print("Status:", m_ripasso.Status, "SolCount:", m_ripasso.SolCount)
if m_ripasso.SolCount > 0:
    valori = np.array([x_ripasso[1].X, x_ripasso[2].X])
    assert np.isfinite(valori).all() and np.isfinite(m_ripasso.ObjVal)
    assert (valori >= -TOL).all() and (np.abs(valori - np.rint(valori)) <= TOL).all()
    assert 2*valori[0] + valori[1] <= 8 + TOL
    assert valori[0] + 2*valori[1] <= 8 + TOL
    assert np.isclose(3*valori[0] + 5*valori[1], m_ripasso.ObjVal, atol=TOL, rtol=0)
    print("Quantità:", valori, "Profitto:", m_ripasso.ObjVal)

m_ripasso.X, m_ripasso.ObjVal

Restricted license - for non-production use only - expires 2027-11-29
Status: 2 SolCount: 2
Quantità: [2. 3.] Profitto: 21.0


([2.0, 3.0], 21.0)

<a id="csv"></a>
## 1. pandas: salvare e leggere i dati

I quattro modelli leggeranno **costi/profitti, coefficienti tecnici, capacità e limiti**
da `dati_lab02/`. I risultati andranno in `risultati_lab02/`.
`Path` gestisce i percorsi; `/` unisce cartella e nome del file;
`mkdir(exist_ok=True)` crea una cartella se manca.

**Il pacchetto ZIP contiene già i CSV.** Se aprite soltanto il notebook, anche in Colab,
la cella di inizializzazione li crea con pandas.

In [5]:
DATI = Path("dati_lab02")
RISULTATI = Path("risultati_lab02")
DATI.mkdir(exist_ok=True)
RISULTATI.mkdir(exist_ok=True)

In [6]:
#@title Preparazione dei CSV (eseguire in caso non sia possibile leggere dati dai CSV)
dati_iniziali = {
  'raffineria_processi.csv': {'impianto': ['I1', 'I2', 'I3'],
                             'greggio_A': [4, 3, 3],
                             'greggio_B': [3, 4, 3],
                             'verde': [2, 4, 2],
                             'super': [3, 2, 2]},
 'raffineria_greggi.csv': {'greggio': ['greggio_A', 'greggio_B'], 'disponibilita_barili': [5000, 6000]},
 'raffineria_prezzi.csv': {'prodotto': ['verde', 'super'], 'prezzo_euro_barile': [120, 150]},
 'agricoltura_tenute.csv': {'tenuta': ['A', 'B', 'C'],
                            'terreno_ha': [600, 700, 450],
                            'acqua_migliaia_m3': [8000, 5000, 6000]},
 'agricoltura_colture.csv': {'coltura': ['mais', 'soia', 'grano'],
                             'resa_migliaia_euro_ha': [5, 6, 7],
                             'acqua_migliaia_m3_ha': [20, 10, 10]},
 'agricoltura_regole.csv': {'parametro': ['quota_massima_soia'], 'valore': [0.4]},
 'parquet_prodotti.csv': {'prodotto': ['Supereconomy',
                                       'Economy',
                                       'Biocity',
                                       'Vintage',
                                       'Gaia',
                                       'Family',
                                       'Extralux'],
                          'profitto_euro_unita': [10, 18, 20, 25, 27, 28, 35],
                          'domanda_massima': [100, 80, 60, 40, 40, 20, 20]},
 'parquet_reparti.csv': {'reparto': ['Falegnameria',
                                     'Verniciatura',
                                     'Assemblaggio',
                                     'Verifica',
                                     'Imballaggio'],
                         'ore_disponibili': [2000, 1500, 1700, 300, 500]},
 'parquet_tempi.csv': {'reparto': ['Falegnameria',
                                   'Verniciatura',
                                   'Assemblaggio',
                                   'Verifica',
                                   'Imballaggio'],
                       'Supereconomy': [7, 2, 2, 1, 1],
                       'Economy': [5, 2, 2, 1, 1],
                       'Biocity': [9, 2, 4, 1, 1],
                       'Vintage': [10, 3, 7, 2, 1],
                       'Gaia': [10, 3, 9, 1, 2],
                       'Family': [12, 3, 15, 2, 1],
                       'Extralux': [15, 3, 18, 2, 0]},
 'trasporti_stabilimenti.csv': {'stabilimento': ['A', 'B', 'C'], 'disponibilita': [50, 70, 20]},
 'trasporti_magazzini.csv': {'magazzino': ['M1', 'M2', 'M3', 'M4'], 'richiesta': [10, 60, 30, 40]},
 'trasporti_costi.csv': {'stabilimento': ['A', 'B', 'C'],
                         'M1': [6, 2, 2],
                         'M2': [8, 3, 4],
                         'M3': [3, 1, 6],
                         'M4': [4, 3, 5]},
  'dieta_alimenti.csv': {'alimento': ['verdura', 'carne', 'frutta'],
                        'costo_euro_kg': [4, 10, 7],
                        'proteine': [5, 15, 4],
                        'ferro': [6, 10, 5],
                        'vitamine': [5, 3, 12]},
 'dieta_fabbisogni.csv': {'nutriente': ['proteine', 'ferro', 'vitamine'], 'minimo_mg': [20, 30, 30]},
 'portfolio_fondi.csv': {'fondo': ['A', 'B', 'C', 'D', 'E'],
                         'tipo': ['privato', 'pubblico', 'stato', 'stato', 'privato'],
                         'rating': ['Aa', 'A', 'Aaa', 'Baa', 'Ba'],
                         'durata_anni': [9, 15, 4, 3, 2],
                         'rendita_percento': [4.5, 5.4, 5.1, 4.4, 6.1]},
 'portfolio_rating.csv': {'rating': ['Aaa', 'Aa', 'A', 'Baa', 'Ba'], 'rischio': [1, 2, 3, 4, 5]},
 'portfolio_parametri.csv': {'parametro': ['capitale',
                                           'aliquota_pubblico_stato',
                                           'quota_min_pubblico_stato',
                                           'durata_media_max',
                                           'rischio_medio_max',
                                           'soglia_A_per_E'],
                             'valore': [100000, 0.3, 0.4, 5, 1.5, 10000]}
  }

for nome_file, colonne in dati_iniziali.items():
    percorso = DATI / nome_file
    if not percorso.exists():
        pd.DataFrame(colonne).to_csv(percorso, index=False)
print("Dati pronti in:", DATI)
del dati_iniziali  # le sezioni successive useranno esclusivamente i file CSV

Dati pronti in: dati_lab02


### Un percorso completo: lettura → tabella → scrittura → rilettura

`read_csv` legge il file; `index_col` sceglie la colonna di etichette.
`to_csv` salva la tabella; `index_label` dà un nome alla colonna dell’indice.
`to_dict` trasforma una Series in un dizionario **ID → valore**.
I CSV usano la virgola come separatore e il punto per i decimali.

In [7]:
prezzi_demo = pd.read_csv(DATI / "raffineria_prezzi.csv", index_col="prodotto")
display(prezzi_demo)
prezzi_demo.to_csv(RISULTATI / "copia_prezzi.csv", index_label="prodotto")
prezzi_riletti = pd.read_csv(RISULTATI / "copia_prezzi.csv", index_col="prodotto")
print("Dizionario dei prezzi:", prezzi_riletti["prezzo_euro_barile"].to_dict())

,prezzo_euro_barile
prodotto,
verde,120
super,150


Dizionario dei prezzi: {'verde': 120, 'super': 150}


**Prova rapida:** nella copia cambiate un prezzo con `.loc["verde", "prezzo_euro_barile"]`,
salvate e rileggete. Cambiare una DataFrame in memoria **non cambia il CSV** finché non
chiamate `to_csv`; cambiare un CSV **non cambia un modello già costruito**.
Occorre rileggere i dati e ricostruire il modello, oppure aggiornare esplicitamente i suoi coefficienti.

Nei modelli useremo `tabella.loc[riga, colonna]` e dizionari indicizzati.
Prima di prodotti matriciali useremo `.loc[ordine_righe, ordine_colonne]`:
l’ordine delle righe nel CSV non deve decidere l’associazione tra coefficienti e variabili.

<a id="dieta"></a>
## 2. Problema 1 — Dieta

Un dietologo deve preparare una dieta, scegliendo tra cibi a base di

- verdura: apporto di 5 mg/kg di proteine, 6 mg/kg di ferro e 5 mg/kg di
vitamine, al costo di 4 euro/kg;
- carne: apporto di 15 mg/kg di proteine, 10 mg/kg di ferro e 3 mg/kg di
vitamine, al costo di 10 euro/kg;
- frutta: apporto di 4 mg/kg di proteine, 5 mg/kg di ferro e 12 mg/kg di
vitamine, al costo di 7 euro/kg.

Determinare la dieta di costo minimo che assicuri un apporto giornaliero di
proteine, ferro e vitamine di almeno 20 mg, 30 mg e 30 mg, rispettivamente.

### Tocca a voi: formulazione

Scrivete significato e dominio delle variabili, obiettivo e vincoli.
Distinguete i dati dalle decisioni e confrontatevi prima di leggere la correzione.

**La vostra formulazione** — modificate questa cella Markdown.

- Variabili e dominio:
- Obiettivo:
- Vincoli:

### Correzione della formulazione

$x_1,x_2,x_3$ sono i **kg** di alimenti
a base di verdura, carne e frutta. Le variabili sono continue non negative.

$$
\begin{aligned}
\min\quad &4x_1+10x_2+7x_3\\
\text{s.t.}\quad &5x_1+15x_2+4x_3\ge20,\\
&6x_1+10x_2+5x_3\ge30,\\
&5x_1+3x_2+12x_3\ge30,\\
&x_1,x_2,x_3\ge0.
\end{aligned}
$$

Siano $A$ gli alimenti e $N$ i nutrienti, $c_j$ il costo per kg, $a_{nj}$ i mg del
nutriente $n$ per kg di alimento $j$, $b_n$ il fabbisogno minimo. In forma indicizzata:
$$
\min\sum_{j\in A}c_jx_j,\qquad
\sum_{j\in A}a_{nj}x_j\ge b_n\quad(n\in N),\qquad x_j\in\mathbb R_{\ge0}.
$$
Il verso è $\ge$: occorre coprire un fabbisogno. Le unità restano quelle della consegna:
$(\mathrm{mg/kg})\cdot\mathrm{kg}=\mathrm{mg}$.

In [8]:
alimenti = pd.read_csv(DATI / "dieta_alimenti.csv", index_col="alimento")
fabbisogni = pd.read_csv(DATI / "dieta_fabbisogni.csv", index_col="nutriente")
A = alimenti.index.tolist()
N = fabbisogni.index.tolist()
assert alimenti.index.is_unique and fabbisogni.index.is_unique
assert set(alimenti.columns) == {"costo_euro_kg"} | set(N)
assert np.isfinite(alimenti.to_numpy()).all() and (alimenti.to_numpy() >= 0).all()
assert np.isfinite(fabbisogni.to_numpy()).all() and (fabbisogni.to_numpy() >= 0).all()
costo_dieta = alimenti["costo_euro_kg"].to_dict()
display(alimenti, fabbisogni)

,costo_euro_kg,proteine,ferro,vitamine
alimento,,,,
verdura,4,5,6,5
carne,10,15,10,3
frutta,7,4,5,12


,minimo_mg
nutriente,
proteine,20
ferro,30
vitamine,30


### Tocca a voi: gurobipy

Usate i dati letti dai CSV. Create variabili, obiettivo e vincoli, poi risolvete.
Usate nomi con suffisso `_stud`. Le righe sotto sono commentate per lasciare il notebook
eseguibile anche se l’esercizio non è stato completato.

In [9]:
# m_dieta_stud = gp.Model("dieta_stud")
# x_dieta_stud = m_dieta_stud.addVars(A, lb=0, vtype=GRB.CONTINUOUS)
# Obiettivo: costo totale; GRB.MINIMIZE.
# Per ogni nutriente n: somma di alimenti.loc[j, n] * x_dieta_stud[j]
# almeno pari a fabbisogni.loc[n, "minimo_mg"].
# Risolvete e controllate SolCount prima di leggere la soluzione.

### Correzione gurobipy

In [10]:
m_dieta = gp.Model("dieta")
m_dieta.Params.OutputFlag = 0
x_dieta = m_dieta.addVars(A, lb=0, vtype=GRB.CONTINUOUS, name="kg")
m_dieta.setObjective(gp.quicksum(costo_dieta[j] * x_dieta[j] for j in A), GRB.MINIMIZE)
for n in N:
    m_dieta.addConstr(gp.quicksum(alimenti.loc[j, n] * x_dieta[j] for j in A)
                     >= fabbisogni.loc[n, "minimo_mg"], name="fabbisogno_" + n)
m_dieta.optimize()
print("Status:", m_dieta.Status, "SolCount:", m_dieta.SolCount)

Status: 2 SolCount: 1


### Analisi del risultato

Ricalcoliamo apporti e costo dalle quantità restituite. Il margine di un vincolo di
fabbisogno è **apporto − fabbisogno**, non negativo entro la tolleranza.
`.loc` riallinea gli ID; `.T` mette i nutrienti sulle righe.

In [11]:
verificato_dieta = None
if m_dieta.SolCount > 0:
    sol_dieta = pd.Series({j: x_dieta[j].X for j in A}, name="kg")
    apporti = alimenti.loc[A, N].T @ sol_dieta.loc[A]
    costo_ricalcolato = float(alimenti.loc[A, "costo_euro_kg"] @ sol_dieta.loc[A])
    assert np.isfinite(sol_dieta.to_numpy()).all() and np.isfinite(m_dieta.ObjVal)
    assert (sol_dieta >= -TOL).all()
    assert (apporti >= fabbisogni.loc[N, "minimo_mg"] - TOL).all()
    assert np.isclose(costo_ricalcolato, m_dieta.ObjVal, atol=TOL, rtol=0)
    verificato_dieta = True
    report_dieta = fabbisogni.copy()
    report_dieta["apporto_mg"] = apporti
    report_dieta["margine_mg"] = apporti - fabbisogni["minimo_mg"]
    report_dieta["vincolo_attivo"] = report_dieta["margine_mg"].abs() <= TOL
    print("Costo minimo (euro):", m_dieta.ObjVal)
    display(sol_dieta.to_frame(), report_dieta)
    sol_dieta.to_csv(RISULTATI / "dieta_piano.csv", index_label="alimento")
    report_dieta.to_csv(RISULTATI / "dieta_nutrienti.csv", index_label="nutriente")
else:
    print("Nessuna soluzione disponibile.")

Costo minimo (euro): 22.340425531914896


,kg
verdura,4.4681
carne,0.0000
frutta,0.6383


,minimo_mg,apporto_mg,margine_mg,vincolo_attivo
nutriente,,,,
proteine,20,24.8936,4.8936,False
ferro,30,30.0000,0.0000,True
vitamine,30,30.0000,0.0000,True


Ferro e vitamine sono esattamente al minimo;
le proteine superano il fabbisogno. Un vincolo $\ge$ non obbliga a raggiungere il minimo
con uguaglianza: la combinazione più economica può dare un eccesso di un nutriente.

<a id="parquet"></a>
## 3. Problema 2 — Ripartizione di risorse

Un’azienda produce vari tipi di parquet che vengono poi venduti sul mercato. Ciascun settore
della produzione può lavorare un numero limitato di ore (mensilmente), come riportato nella
seguente tabella.

| | Falegnameria | Verniciatura | Assemblaggio | Verifica | Imballaggio |
|---|---:|---:|---:|---:|---:|
| Ore | 2000 | 1500 | 1700 | 300 | 500 |

Nella seguente tabella si riporta il profitto unitario e la domanda massima per tipologia di parquet.

| | Supereconomy | Economy | Biocity | Vintage | Gaia | Family | Extralux |
|---|---:|---:|---:|---:|---:|---:|---:|
| Profitto | 10 | 18 | 20 | 25 | 27 | 28 | 35 |
| Domanda | 100 | 80 | 60 | 40 | 40 | 20 | 20 |

Infine, le ore di lavorazione necessarie per un’unità di ciascuna tipologia di parquet sono riportate
nella seguente tabella.

| | Superecon. | Economy | Biocity | Vintage | Gaia | Family | Extralux |
|---|---:|---:|---:|---:|---:|---:|---:|
| Falegnameria | 7 | 5 | 9 | 10 | 10 | 12 | 15 |
| Verniciatura | 2 | 2 | 2 | 3 | 3 | 3 | 3 |
| Assemblaggio | 2 | 2 | 4 | 7 | 9 | 15 | 18 |
| Verifica | 1 | 1 | 1 | 2 | 1 | 2 | 2 |
| Imballaggio | 1 | 1 | 1 | 1 | 2 | 1 | 0 |

Determinare il piano produttivo che massimizza il profitto totale.

### Tocca a voi: formulazione

Scrivete significato e dominio delle variabili, obiettivo e vincoli.
Distinguete i dati dalle decisioni e confrontatevi prima di leggere la correzione.

**La vostra formulazione** — modificate questa cella Markdown.

- Variabili e dominio:
- Obiettivo:
- Vincoli:

### Correzione della formulazione

$P$ è l’insieme dei prodotti, $R$ quello dei reparti;
$c_j$ è il profitto unitario, $d_j$ la domanda massima, $a_{ij}$ le ore di lavorazione,
$b_i$ le ore disponibili. $x_j$ è la quantità da produrre, **continua**.

$$
\begin{aligned}
\max\quad &\sum_{j\in P}c_jx_j\\
\text{s.t.}\quad &\sum_{j\in P}a_{ij}x_j\le b_i &&i\in R,\\
&0\le x_j\le d_j &&j\in P.
\end{aligned}
$$

La domanda è un limite superiore, non un minimo da soddisfare.
Nel codice `J` rappresenta l’insieme $P$. Un dizionario `domanda` passato a
`addVars(..., ub=domanda)` assegna a ogni variabile il suo limite superiore.

In [12]:
prodotti = pd.read_csv(DATI / "parquet_prodotti.csv", index_col="prodotto")
reparti = pd.read_csv(DATI / "parquet_reparti.csv", index_col="reparto")
tempi = pd.read_csv(DATI / "parquet_tempi.csv", index_col="reparto")
J = prodotti.index.tolist()
R = reparti.index.tolist()
assert prodotti.index.is_unique and reparti.index.is_unique and tempi.index.is_unique
assert set(tempi.index) == set(R) and set(tempi.columns) == set(J)
tempi = tempi.loc[R, J]  # riallineamento esplicito per etichette
assert np.isfinite(tempi.to_numpy()).all() and (tempi.to_numpy() >= 0).all()
assert np.isfinite(prodotti.to_numpy()).all() and (prodotti.to_numpy() >= 0).all()
assert np.isfinite(reparti.to_numpy()).all() and (reparti.to_numpy() >= 0).all()
profitto = prodotti["profitto_euro_unita"].to_dict()
domanda = prodotti["domanda_massima"].to_dict()
display(prodotti, reparti, tempi)

,profitto_euro_unita,domanda_massima
prodotto,,
Supereconomy,10,100
Economy,18,80
Biocity,20,60
Vintage,25,40
Gaia,27,40
Family,28,20
Extralux,35,20


,ore_disponibili
reparto,
Falegnameria,2000
Verniciatura,1500
Assemblaggio,1700
Verifica,300
Imballaggio,500


,Supereconomy,Economy,Biocity,Vintage,Gaia,Family,Extralux
reparto,,,,,,,
Falegnameria,7,5,9,10,10,12,15
Verniciatura,2,2,2,3,3,3,3
Assemblaggio,2,2,4,7,9,15,18
Verifica,1,1,1,2,1,2,2
Imballaggio,1,1,1,1,2,1,0


### Tocca a voi: gurobipy

Usate i dati letti dai CSV. Create variabili, obiettivo e vincoli, poi risolvete.
Usate nomi con suffisso `_stud`. Le righe sotto sono commentate per lasciare il notebook
eseguibile anche se l’esercizio non è stato completato.

In [13]:
# m_par_stud = gp.Model("parquet_stud")
# x_par_stud = m_par_stud.addVars(J, lb=0, ub=domanda, vtype=GRB.CONTINUOUS)
# Obiettivo: somma dei profitti.
# Per ogni reparto r: consumo di ore <= reparti.loc[r, "ore_disponibili"].
# Risolvete e ricostruite il consumo di ogni reparto dal piano trovato.

### Correzione gurobipy

In [14]:
m_par = gp.Model("parquet")
m_par.Params.OutputFlag = 0
x_par = m_par.addVars(J, lb=0, ub=domanda, vtype=GRB.CONTINUOUS, name="q")
m_par.setObjective(gp.quicksum(profitto[j] * x_par[j] for j in J), GRB.MAXIMIZE)
for r in R:
    m_par.addConstr(gp.quicksum(tempi.loc[r, j] * x_par[j] for j in J)
                   <= reparti.loc[r, "ore_disponibili"], name="ore_" + r)
m_par.optimize()
print("Status:", m_par.Status, "SolCount:", m_par.SolCount)

Status: 2 SolCount: 1


### Analisi del risultato

Un vincolo di capacità ha margine **disponibilità − uso**. Verifichiamo anche i limiti
sulle singole quantità e ricalcoliamo l’obiettivo senza arrotondare i valori del solver.

In [15]:
verificato_par = None
if m_par.SolCount > 0:
    sol_par = pd.Series({j: x_par[j].X for j in J}, name="produzione")
    uso_par = tempi.loc[R, J] @ sol_par.loc[J]
    valore_par = float(prodotti.loc[J, "profitto_euro_unita"] @ sol_par.loc[J])
    assert np.isfinite(sol_par.to_numpy()).all() and np.isfinite(m_par.ObjVal)
    assert (sol_par >= -TOL).all()
    assert (sol_par <= prodotti.loc[J, "domanda_massima"] + TOL).all()
    assert (uso_par <= reparti.loc[R, "ore_disponibili"] + TOL).all()
    assert np.isclose(valore_par, m_par.ObjVal, atol=TOL, rtol=0)
    verificato_par = True
    piano_par = prodotti.copy()
    piano_par["produzione"] = sol_par
    piano_par["domanda_non_coperta"] = piano_par["domanda_massima"] - sol_par
    report_par = reparti.copy()
    report_par["ore_usate"] = uso_par
    report_par["ore_residue"] = report_par["ore_disponibili"] - uso_par
    report_par["vincolo_attivo"] = report_par["ore_residue"].abs() <= TOL
    print("Profitto mensile (unità di profitto):", m_par.ObjVal)
    display(piano_par, report_par)
    piano_par.to_csv(RISULTATI / "parquet_piano.csv", index_label="prodotto")
    report_par.to_csv(RISULTATI / "parquet_risorse.csv", index_label="reparto")
else:
    print("Nessuna soluzione disponibile.")

Profitto mensile (unità di profitto): 5340.0


,profitto_euro_unita,domanda_massima,produzione,domanda_non_coperta
prodotto,,,,
Supereconomy,10,100,0.0000,100.0000
Economy,18,80,80.0000,0.0000
Biocity,20,60,46.6667,13.3333
Vintage,25,40,40.0000,0.0000
Gaia,27,40,40.0000,0.0000
Family,28,20,6.6667,13.3333
Extralux,35,20,20.0000,0.0000


,ore_disponibili,ore_usate,ore_residue,vincolo_attivo
reparto,,,,
Falegnameria,2000,2000.0000,0.0000,True
Verniciatura,1500,573.3333,926.6667,False
Assemblaggio,1700,1446.6667,253.3333,False
Verifica,300,300.0000,0.0000,True
Imballaggio,500,253.3333,246.6667,False


Con i dati originali, un piano ottimo nell’ordine della tabella è
$(0,80,140/3,40,40,20/3,20)$, con profitto **5340**.
Falegnameria e Verifica sono sature. Il prodotto con il profitto unitario maggiore
non è necessariamente quello da produrre in quantità maggiore: contano anche i tempi
di lavorazione e la domanda massima.

<a id="trasporti"></a>
## 4. Problema 3 — Trasporti

Una ditta produce elettrodomestici in tre stabilimenti e li smista in quattro
magazzini intermedi di vendita. La produzione settimanale nei tre stabilimenti
A, B e C è rispettivamente 50, 70 e 20 unità.

La quantità richiesta dai quattro magazzini è rispettivamente di 10, 60, 30 e
40 unità.

I costi per il trasporto di un elettrodomestico tra gli stabilimenti e i magazzini
sono i seguenti:
- dallo stabilimento A: 6, 8, 3 e 4 euro;
- dallo stabilimento B: 2, 3, 1 e 3 euro;
- dallo stabilimento C: 2, 4, 6 e 5 euro.

Determinare il piano di trasporti di costo minimo.

### Tocca a voi: formulazione

Scrivete significato e dominio delle variabili, obiettivo e vincoli.
Distinguete i dati dalle decisioni e confrontatevi prima di leggere la correzione.

**La vostra formulazione** — modificate questa cella Markdown.

- Variabili e dominio:
- Obiettivo:
- Vincoli:

### Correzione della formulazione

$x_{ij}\in\mathbb Z_{\ge0}$ è il numero di elettrodomestici spediti
dallo stabilimento $i\in S$ al magazzino $j\in M$. Siano $s_i$ le disponibilità,
$d_j$ le richieste e $c_{ij}$ i costi per unità.

$$
\begin{aligned}
\min\quad &\sum_{i\in S}\sum_{j\in M}c_{ij}x_{ij}\\
\text{s.t.}\quad &\sum_{j\in M}x_{ij}\le s_i &&i\in S,\\
&\sum_{i\in S}x_{ij}\ge d_j &&j\in M,\\
&x_{ij}\in\mathbb Z_{\ge0} &&i\in S,\ j\in M.
\end{aligned}
$$

Poiché offerta e domanda totali valgono
entrambe 140, per questi dati i vincoli sono necessariamente soddisfatti con uguaglianza.
Nei CSV gli ID `M1`, …, `M4` identificano i magazzini 1, …, 4 della consegna.

`addVars(S, M, ...)` crea una variabile per ogni coppia. Il primo indice è l’origine,
il secondo la destinazione; una doppia somma è `gp.quicksum(... for i in S for j in M)`.

In [16]:
stabilimenti = pd.read_csv(DATI / "trasporti_stabilimenti.csv", index_col="stabilimento")
magazzini = pd.read_csv(DATI / "trasporti_magazzini.csv", index_col="magazzino")
costi = pd.read_csv(DATI / "trasporti_costi.csv", index_col="stabilimento")
S = stabilimenti.index.tolist()
M = magazzini.index.tolist()
assert stabilimenti.index.is_unique and magazzini.index.is_unique and costi.index.is_unique
assert set(costi.index) == set(S) and set(costi.columns) == set(M)
costi = costi.loc[S, M]
assert np.isfinite(costi.to_numpy()).all() and (costi.to_numpy() >= 0).all()
assert np.isfinite(stabilimenti.to_numpy()).all() and (stabilimenti.to_numpy() >= 0).all()
assert np.isfinite(magazzini.to_numpy()).all() and (magazzini.to_numpy() >= 0).all()
assert (stabilimenti["disponibilita"] % 1 == 0).all() and (magazzini["richiesta"] % 1 == 0).all()
display(stabilimenti, magazzini, costi)
print("Disponibilità totale:", stabilimenti["disponibilita"].sum(),
      "Richiesta totale:", magazzini["richiesta"].sum())

,disponibilita
stabilimento,
A,50
B,70
C,20


,richiesta
magazzino,
M1,10
M2,60
M3,30
M4,40


,M1,M2,M3,M4
stabilimento,,,,
A,6,8,3,4
B,2,3,1,3
C,2,4,6,5


Disponibilità totale: 140 Richiesta totale: 140


### Tocca a voi: gurobipy

Usate i dati letti dai CSV. Create variabili, obiettivo e vincoli, poi risolvete.
Usate nomi con suffisso `_stud`. Le righe sotto sono commentate per lasciare il notebook
eseguibile anche se l’esercizio non è stato completato.

In [17]:
# m_tra_stud = gp.Model("trasporti_stud")
# x_tra_stud = m_tra_stud.addVars(S, M, lb=0, vtype=GRB.INTEGER)
# Obiettivo: costo totale; GRB.MINIMIZE.
# Un vincolo sulle partenze per ogni stabilimento.
# Un vincolo sugli arrivi per ogni magazzino.
# Risolvete e controllate anche l'integralità delle quantità.

### Correzione gurobipy

In [18]:
m_tra = gp.Model("trasporti")
m_tra.Params.OutputFlag = 0
x_tra = m_tra.addVars(S, M, lb=0, vtype=GRB.INTEGER, name="spediti")
m_tra.setObjective(gp.quicksum(costi.loc[i, j] * x_tra[i, j]
                              for i in S for j in M), GRB.MINIMIZE)
for i in S:
    m_tra.addConstr(gp.quicksum(x_tra[i, j] for j in M)
                   <= stabilimenti.loc[i, "disponibilita"], name="offerta_" + i)
for j in M:
    m_tra.addConstr(gp.quicksum(x_tra[i, j] for i in S)
                   >= magazzini.loc[j, "richiesta"], name="domanda_" + j)
m_tra.optimize()
print("Status:", m_tra.Status, "SolCount:", m_tra.SolCount)

Status: 2 SolCount: 2


### Analisi del risultato

Le somme per riga sono le partenze; quelle per colonna sono gli arrivi. Verifichiamo
domini, disponibilità, richieste e costo, poi salviamo il piano nei CSV dei risultati.

In [19]:
verificato_tra = None
if m_tra.SolCount > 0:
    piano_tra = pd.DataFrame([[x_tra[i, j].X for j in M] for i in S], index=S, columns=M)
    partenze = piano_tra.sum(axis=1)
    arrivi = piano_tra.sum(axis=0)
    valore_tra = float((costi.loc[S, M] * piano_tra.loc[S, M]).to_numpy().sum())
    valori_tra = piano_tra.to_numpy()
    assert np.isfinite(valori_tra).all() and np.isfinite(m_tra.ObjVal)
    assert (valori_tra >= -TOL).all()
    assert (np.abs(valori_tra - np.rint(valori_tra)) <= TOL).all()
    assert (partenze <= stabilimenti.loc[S, "disponibilita"] + TOL).all()
    assert (arrivi >= magazzini.loc[M, "richiesta"] - TOL).all()
    assert np.isclose(valore_tra, m_tra.ObjVal, atol=TOL, rtol=0)
    verificato_tra = True
    report_partenze = stabilimenti.copy()
    report_partenze["spediti"] = partenze
    report_partenze["rimanenti"] = stabilimenti["disponibilita"] - partenze
    report_arrivi = magazzini.copy()
    report_arrivi["ricevuti"] = arrivi
    report_arrivi["eccedenza"] = arrivi - magazzini["richiesta"]
    print("Costo settimanale (euro):", m_tra.ObjVal)
    display(piano_tra, report_partenze, report_arrivi)
    piano_tra.to_csv(RISULTATI / "trasporti_piano.csv", index_label="stabilimento")
    report_partenze.to_csv(RISULTATI / "trasporti_partenze.csv", index_label="stabilimento")
    report_arrivi.to_csv(RISULTATI / "trasporti_arrivi.csv", index_label="magazzino")
else:
    print("Nessuna soluzione disponibile: controllare stato, disponibilità e richieste.")

Costo settimanale (euro): 420.0


,M1,M2,M3,M4
A,-0.0,-0.0,10.0,40.0
B,-0.0,50.0,20.0,-0.0
C,10.0,10.0,-0.0,-0.0


,disponibilita,spediti,rimanenti
stabilimento,,,
A,50,50.0,0.0
B,70,70.0,0.0
C,20,20.0,0.0


,richiesta,ricevuti,eccedenza
magazzino,,,
M1,10,10.0,0.0
M2,60,60.0,0.0
M3,30,30.0,0.0
M4,40,40.0,0.0


Con i dati originali, il costo ottimo è **420 euro**. Le disponibilità vengono utilizzate
interamente e tutte le richieste sono coperte esattamente. Possono esistere più piani
ottimi: conta che vincoli e costo siano corretti, non che ogni cella coincida con quella
del docente. Un eventuale `-0.0` è lo zero numerico.

<a id="portfolio"></a>
## 5. Problema 4 — Portfolio optimization

Un cliente affida ad un’agenzia finanziaria 100 000 euro da impiegare in fondi
di investimento. I fondi attualmente offerti dal mercato sono cinque, come
riassunto in tabella:

| Nome | Tipo | Moody’s rating | Durata (anni) | Rendita alla maturazione |
|---|---|---|---:|---:|
| A | privato | Aa | 9 | 4,5% |
| B | pubblico | A | 15 | 5,4% |
| C | stato | Aaa | 4 | 5,1% |
| D | stato | Baa | 3 | 4,4% |
| E | privato | Ba | 2 | 6,1% |

Si sa che i fondi pubblici e dello stato sono tassati del 30% alla fine del
periodo. Il cliente chiede di riservare almeno il 40% del capitale a fondi
pubblici e dello stato e vuole che la durata media dell’investimento non superi
i 5 anni. Inoltre, trasformando il Moody’s rating in una scala numerica (Aaa =
1, Aa = 2, A = 3, Baa = 4 e Ba = 5), il valore medio del rischio
dell’investimento non deve superare 1,5. Si vuole massimizzare la rendita
finale dell’investimento.\

La durata media e il rischio medio sono ponderati con gli importi investiti nei rispettivi fondi.

### Tocca a voi: formulazione

Scrivete significato e dominio delle variabili, obiettivo e vincoli.
Distinguete i dati dalle decisioni e confrontatevi prima di leggere la correzione.

**La vostra formulazione** — modificate questa cella Markdown.

- Variabili e dominio:
- Obiettivo:
- Vincoli:

### Correzione della formulazione

$x_A,\ldots,x_E\ge0$ sono gli **euro investiti** nei cinque fondi.

$$
\begin{aligned}
\max\quad &4{,}5x_A+0{,}7\cdot5{,}4x_B+0{,}7\cdot5{,}1x_C
                +0{,}7\cdot4{,}4x_D+6{,}1x_E\\
\text{s.t.}\quad &x_A+x_B+x_C+x_D+x_E\le100000,\\
&2x_A+3x_B+x_C+4x_D+5x_E\le1{,}5(x_A+x_B+x_C+x_D+x_E),\\
&9x_A+15x_B+4x_C+3x_D+2x_E\le5(x_A+x_B+x_C+x_D+x_E),\\
&x_B+x_C+x_D\ge40000,\\
&x_i\in\mathbb R_{\ge0}\qquad i\in\{A,B,C,D,E\}.
\end{aligned}
$$

**Medie ponderate.** Il rischio medio è $\sum_i s_ix_i/\sum_i x_i$; la durata media è
$\sum_i t_ix_i/\sum_i x_i$. Il denominatore è il capitale **effettivamente investito**,
non necessariamente 100000. Poiché almeno 40000 euro vanno ai fondi pubblici/statali,
il denominatore è positivo: moltiplicando si ottengono i vincoli lineari della slide.
La quota minima di 40000 euro è invece il 40% del **capitale disponibile**.

### Lettura dei dati e coefficienti derivati

`Series.map` associa a ciascun rating il numero della scala letta dal CSV.
`isin(["pubblico", "stato"])` identifica i fondi soggetti a tassazione e quota minima.
La colonna `rendita_netta_percento` resta nella scala percentuale della slide.

In [20]:
fondi = pd.read_csv(DATI / "portfolio_fondi.csv", index_col="fondo")
scala_rating = pd.read_csv(DATI / "portfolio_rating.csv", index_col="rating")
parametri = pd.read_csv(DATI / "portfolio_parametri.csv", index_col="parametro")["valore"]
F = fondi.index.tolist()
assert fondi.index.is_unique and scala_rating.index.is_unique and parametri.index.is_unique
assert set(fondi["rating"]) <= set(scala_rating.index)
assert set(fondi["tipo"]) <= {"privato", "pubblico", "stato"}
fondi["rischio"] = fondi["rating"].map(scala_rating["rischio"])
pubblici = fondi.index[fondi["tipo"].isin(["pubblico", "stato"])].tolist()
fondi["aliquota"] = 0.0
fondi.loc[pubblici, "aliquota"] = parametri["aliquota_pubblico_stato"]
fondi["rendita_netta_percento"] = fondi["rendita_percento"] * (1 - fondi["aliquota"])
capitale = float(parametri["capitale"])
quota_min = float(parametri["quota_min_pubblico_stato"])
rischio_max = float(parametri["rischio_medio_max"])
durata_max = float(parametri["durata_media_max"])
numeri = fondi[["durata_anni", "rendita_percento", "rischio", "aliquota", "rendita_netta_percento"]]
assert np.isfinite(numeri.to_numpy()).all() and (numeri.to_numpy() >= 0).all()
assert np.isfinite(parametri.to_numpy()).all() and capitale > 0
assert 0 < quota_min <= 1 and 0 <= parametri["aliquota_pubblico_stato"] <= 1
display(fondi, parametri.to_frame())

,tipo,rating,durata_anni,rendita_percento,rischio,aliquota,rendita_netta_percento
fondo,,,,,,,
A,privato,Aa,9,4.5,2,0.0,4.50
B,pubblico,A,15,5.4,3,0.3,3.78
C,stato,Aaa,4,5.1,1,0.3,3.57
D,stato,Baa,3,4.4,4,0.3,3.08
E,privato,Ba,2,6.1,5,0.0,6.10


,valore
parametro,
capitale,100000.0
aliquota_pubblico_stato,0.3
quota_min_pubblico_stato,0.4
durata_media_max,5.0
rischio_medio_max,1.5
soglia_A_per_E,10000.0


### Tocca a voi: gurobipy

Usate i dati letti dai CSV. Create variabili, obiettivo e vincoli, poi risolvete.
Usate nomi con suffisso `_stud`. Le righe sotto sono commentate per lasciare il notebook
eseguibile anche se l’esercizio non è stato completato.

In [21]:
# m_port_stud = gp.Model("portfolio_stud")
# x_port_stud = m_port_stud.addVars(F, lb=0, vtype=GRB.CONTINUOUS)
# Definite il capitale investito come gp.quicksum(x_port_stud[i] for i in F).
# Obiettivo: somma di rendita_netta_percento * investimento.
# Aggiungete budget, rischio medio, durata media e quota minima pubblica/statale.
# I vincoli di media hanno il totale investito anche a destra.

### Correzione gurobipy

In [22]:
m_port = gp.Model("portfolio")
m_port.Params.OutputFlag = 0
x_port = m_port.addVars(F, lb=0, vtype=GRB.CONTINUOUS, name="investimento")
investito = gp.quicksum(x_port[i] for i in F)
m_port.setObjective(gp.quicksum(fondi.loc[i, "rendita_netta_percento"] * x_port[i]
                               for i in F), GRB.MAXIMIZE)
m_port.addConstr(investito <= capitale, name="capitale")
m_port.addConstr(gp.quicksum(fondi.loc[i, "rischio"] * x_port[i] for i in F)
                 <= rischio_max * investito, name="rischio_medio")
m_port.addConstr(gp.quicksum(fondi.loc[i, "durata_anni"] * x_port[i] for i in F)
                 <= durata_max * investito, name="durata_media")
m_port.addConstr(gp.quicksum(x_port[i] for i in pubblici)
                 >= quota_min * capitale, name="quota_pubblica")
m_port.optimize()
print("Status:", m_port.Status, "SolCount:", m_port.SolCount)

Status: 2 SolCount: 1


### Analisi del risultato

Controlliamo i vincoli nella forma lineare originaria; poi calcoliamo le medie per
interpretare il piano. Sono ponderate con gli importi investiti, non con il numero dei
fondi. Ogni margine nella tabella deve essere non negativo entro `TOL`.

In [23]:
verificato_port = None
if m_port.SolCount > 0:
    sol_port = pd.Series({i: x_port[i].X for i in F}, name="investimento_euro")
    totale_port = float(sol_port.sum())
    totale_pubblico = float(sol_port.loc[pubblici].sum())
    somma_rischio = float(fondi.loc[F, "rischio"] @ sol_port.loc[F])
    somma_durata = float(fondi.loc[F, "durata_anni"] @ sol_port.loc[F])
    valore_port = float(fondi.loc[F, "rendita_netta_percento"] @ sol_port.loc[F])
    margini_port = pd.Series({"capitale (euro)": capitale - totale_port,
        "rischio (punteggio × euro)": rischio_max * totale_port - somma_rischio,
        "durata (anni × euro)": durata_max * totale_port - somma_durata,
        "quota pubblica/statale (euro)": totale_pubblico - quota_min * capitale})
    assert np.isfinite(sol_port.to_numpy()).all() and np.isfinite(m_port.ObjVal)
    assert (sol_port >= -TOL).all() and (margini_port >= -TOL).all()
    assert np.isclose(valore_port, m_port.ObjVal, atol=TOL, rtol=0)
    verificato_port = True
    piano_port = fondi.copy()
    piano_port["investimento_euro"] = sol_port
    piano_port["rendita_netta_euro"] = fondi["rendita_netta_percento"] * sol_port / 100
    print("Obiettivo nella scala della slide:", m_port.ObjVal)
    print("Rendita netta in euro:", m_port.ObjVal / 100)
    if totale_port > TOL:
        print("Rischio medio:", somma_rischio / totale_port,
              "Durata media:", somma_durata / totale_port)
    display(piano_port, margini_port.to_frame("Margine"))
    piano_port.to_csv(RISULTATI / "portfolio_piano.csv", index_label="fondo")
    margini_port.to_csv(RISULTATI / "portfolio_margini.csv", index_label="vincolo")
else:
    print("Nessuna soluzione disponibile.")

Obiettivo nella scala della slide: 395386.3636363636
Rendita netta in euro: 3953.863636363636
Rischio medio: 1.5 Durata media: 5.0


,tipo,rating,durata_anni,rendita_percento,rischio,aliquota,rendita_netta_percento,investimento_euro,rendita_netta_euro
fondo,,,,,,,,,
A,privato,Aa,9,4.5,2,0.0,4.50,22727.2727,1022.7273
B,pubblico,A,15,5.4,3,0.3,3.78,0.0000,0.0000
C,stato,Aaa,4,5.1,1,0.3,3.57,70454.5455,2515.2273
D,stato,Baa,3,4.4,4,0.3,3.08,0.0000,0.0000
E,privato,Ba,2,6.1,5,0.0,6.10,6818.1818,415.9091


,Margine
capitale (euro),0.0000
rischio (punteggio × euro),0.0000
durata (anni × euro),0.0000
quota pubblica/statale (euro),30454.5455


Con i dati originali, gli importi sono circa **22.727,27 euro in A**, **70.454,55 in C**,
**6.818,18 in E**, zero in B e D. Sono investiti tutti i 100.000 euro;
rischio medio e durata media sono rispettivamente **1,5** e **5 anni**.
La rendita netta è circa **3.953,86 euro**.

Perché non investire tutto in E, che ha la rendita maggiore? Violerebbe rischio e quota
pubblica/statale.

## 7. Facoltativo — Portfolio optimization: estensioni

Modificare opportunamente il modello per prendere in considerazione le due
seguenti condizioni aggiuntive:

- le regole del mercato impongono che l’investimento in C precluda la
possibilità di investire in D, e viceversa;
- è possibile investire in E solo se si sono investiti almeno 10 000 euro in A.

### Formulazione aggiuntiva

Ai vincoli precedenti aggiungiamo variabili binarie $y_C,y_D,y_E$ e:
$$
\begin{aligned}
x_C&\le My_C, & x_D&\le My_D, & y_C+y_D&\le1,\\
x_E&\le My_E, & x_A&\ge10000y_E,
\end{aligned}
\qquad y_C,y_D,y_E\in\{0,1\}.
$$
È sufficiente $M=100000$, perché non negatività e budget implicano $x_i\le100000$.
Il primo gruppo impedisce investimenti contemporanei in C e D. Se $x_E>0$, il secondo
impone $y_E=1$ e quindi almeno 10000 euro in A. Non impone l’implicazione inversa.
Le variabili $x_i$ restano continue; servono binarie solo per C, D ed E.

`copy()` copia il modello già risolto; `getVarByName` ritrova le sue variabili.
Il modello base rimane disponibile per confrontare gli obiettivi.

In [24]:
m_logico = m_port.copy()
m_logico.Params.OutputFlag = 0
x_logico = {i: m_logico.getVarByName("investimento[" + i + "]") for i in F}
y_logico = m_logico.addVars(["C", "D", "E"], vtype=GRB.BINARY, name="attivo")
grande_M = capitale
soglia_A = float(parametri["soglia_A_per_E"])
m_logico.addConstr(x_logico["C"] <= grande_M * y_logico["C"], name="attivazione_C")
m_logico.addConstr(x_logico["D"] <= grande_M * y_logico["D"], name="attivazione_D")
m_logico.addConstr(y_logico["C"] + y_logico["D"] <= 1, name="esclusione_CD")
m_logico.addConstr(x_logico["E"] <= grande_M * y_logico["E"], name="attivazione_E")
m_logico.addConstr(x_logico["A"] >= soglia_A * y_logico["E"], name="implicazione_E_A")
m_logico.optimize()
print("Status:", m_logico.Status, "SolCount:", m_logico.SolCount)

Status: 2 SolCount: 1


In [25]:
if m_logico.SolCount > 0:
    sol_logica = pd.Series({i: x_logico[i].X for i in F})
    binarie = pd.Series({i: y_logico[i].X for i in ["C", "D", "E"]})
    totale_logico = float(sol_logica.sum())
    assert np.isfinite(sol_logica.to_numpy()).all() and np.isfinite(m_logico.ObjVal)
    assert np.isfinite(binarie.to_numpy()).all()
    assert (sol_logica >= -TOL).all() and totale_logico <= capitale + TOL
    assert (binarie >= -TOL).all() and (binarie <= 1 + TOL).all()
    assert (np.abs(binarie - np.rint(binarie)) <= TOL).all()
    assert fondi.loc[F, "rischio"] @ sol_logica.loc[F] <= rischio_max * totale_logico + TOL
    assert fondi.loc[F, "durata_anni"] @ sol_logica.loc[F] <= durata_max * totale_logico + TOL
    assert sol_logica.loc[pubblici].sum() >= quota_min * capitale - TOL
    assert sol_logica["C"] <= grande_M * binarie["C"] + TOL
    assert sol_logica["D"] <= grande_M * binarie["D"] + TOL
    assert binarie["C"] + binarie["D"] <= 1 + TOL
    assert sol_logica["E"] <= grande_M * binarie["E"] + TOL
    assert sol_logica["A"] >= soglia_A * binarie["E"] - TOL
    assert np.isclose(fondi.loc[F, "rendita_netta_percento"] @ sol_logica.loc[F],
                      m_logico.ObjVal, atol=TOL, rtol=0)
    display(sol_logica.to_frame("Investimento (euro)"), binarie.to_frame("Variabile binaria"))
    print("Rendita con i vincoli logici (euro):", m_logico.ObjVal / 100)
    if m_port.Status == GRB.OPTIMAL and m_logico.Status == GRB.OPTIMAL and m_port.SolCount > 0:
        print("Variazione rispetto al modello base (euro):", (m_logico.ObjVal - m_port.ObjVal) / 100)
    sol_logica.to_csv(RISULTATI / "portfolio_estensione.csv", index_label="fondo")

,Investimento (euro)
A,22727.2727
B,0.0000
C,70454.5455
D,0.0000
E,6818.1818


,Variabile binaria
C,1.0
D,0.0
E,1.0


Rendita con i vincoli logici (euro): 3953.863636363635
Variazione rispetto al modello base (euro): -5.820766091346741e-13


Con i dati originali la rendita ottima resta invariata: il piano base investe già in C
e non in D, e l’investimento in A supera 10000 euro. Aggiungere vincoli restringe l’insieme
ammissibile, ma non peggiora necessariamente il valore ottimo se un ottimo precedente
rimane ammissibile.